In [ ]:
from astropy import wcs
from astropy.io import fits
import numpy as np
import matplotlib as plt
import galsim
from astropy.wcs import WCS
import sys

sys.path.append("src")

### This is a Jupyter Notebook that has been helping me with my debugging for `test_image_sim.py` and also for `imagesim.py`.

Debugging test that fixed `test_convert_pos` function from `test_image_sim.py` that fixed wcs argument.

In [ ]:
myheader = fits.Header.fromtextfile(
    "/users/PAS2340/karadiludovico/sub_pixel_response/tests/sub_pixel_response/test_wcs_txt"
)

In [ ]:
wcsstring = """
XTENSION= 'IMAGE   '           / Image extension                                
BITPIX  =                  -64 / array data type                                
NAXIS   =                    2 / number of array dimensions                     
NAXIS1  =                 4088                                                  
NAXIS2  =                 4088                                                  
PCOUNT  =                    0 / number of parameters                           
GCOUNT  =                    1 / number of groups                               
EXPTIME =                139.8                                                  
MJD-OBS =         62471.492045                                                  
DATE-OBS= '2029-12-01 11:48:32.688000'                                          
FILTER  = 'H158    '                                                            
ZPTMAG  =   16.800870916182618                                                  
GS_XMIN =                    1 / GalSim image minimum x coordinate              
GS_YMIN =                    1 / GalSim image minimum y coordinate              
GS_WCS  = 'GSFitsWCS'          / GalSim WCS name                                
CTYPE1  = 'RA---TAN-SIP'                                                        
CTYPE2  = 'DEC--TAN-SIP'                                                        
CRPIX1  =               2044.0                                                  
CRPIX2  =               2044.0                                                  
CD1_1   = 3.01922901086850E-05                                                  
CD1_2   = 1.45559107465136E-06                                                  
CD2_1   = -8.3872456214526E-07                                                  
CD2_2   = 2.93576778237758E-05                                                  
CUNIT1  = 'deg     '                                                            
CUNIT2  = 'deg     '                                                            
CRVAL1  =   10.208584415642562                                                  
CRVAL2  =   -44.33853770184239                                                  
A_ORDER =                    4                                                  
A_0_2   =      3.851828071E-10                                                  
A_0_3   =      5.492409696E-14                                                  
A_0_4   =      3.825353128E-18                                                  
A_1_1   =     -1.232185377E-09                                                  
A_1_2   =     -9.743979693E-14                                                  
A_1_3   =       2.66249338E-17                                                  
A_2_0   =      3.802404353E-10                                                  
A_2_1   =     -9.031463862E-14                                                  
A_2_2   =     -6.271302544E-17                                                  
A_3_0   =      2.325088216E-14                                                  
A_3_1   =      2.521067326E-17                                                  
A_4_0   =      1.425534054E-17                                                  
B_ORDER =                    4                                                  
B_0_2   =     -1.175573884E-09                                                  
B_0_3   =      1.303875779E-14                                                  
B_0_4   =      1.602230927E-17                                                  
B_1_1   =     -1.793186122E-11                                                  
B_1_2   =     -1.532973486E-13                                                  
B_1_3   =     -3.870326104E-17                                                  
B_2_0   =      5.982538571E-11                                                  
B_2_1   =      1.443685076E-13                                                  
B_2_2   =      1.727380843E-17                                                  
B_3_0   =      2.897221014E-14                                                  
B_3_1   =      2.713725388E-17                                                  
B_4_0   =      1.591294122E-18                                                  
EQUINOX =               2000.0                                                  
WCSAXES =                    2                                                  
WCSNAME = 'wfiwcs_20210204_d2'                                                  
TELESCOP= 'Roman   '                                                            
INSTRUME= 'WFC     '                                                            
RA_TARG =               10.489                                                  
DEC_TARG=             -44.4299                                                  
PA_OBSY =  -118.00999999999999                                                  
PA_FPA  =   1.9899999999999998                                                  
SCA_NUM =                   14                                                  
ORIENTAT=   2.1863008787824225                                                  
LONPOLE =                180.0                                                  
SKY_MEAN=                 74.0                                                  
EXTNAME = 'SCI     '           / extension name                                 
EXTVER  =                    1 / extension value                                
"""

Testing to see if the `convert_pos` function worked. 

In [ ]:
from sub_pixel_response.imagesim import convert_pos

In [ ]:
myheader = fits.Header.fromstring(wcsstring, sep="\n")
mywcs = galsim.AstropyWCS(header=myheader)
ra = 10.208584415642562 * galsim.degrees
dec = -44.33853770184239 * galsim.degrees
x, y = convert_pos(ra, dec, mywcs)
expected_x = 2044.0
expected_y = 2044.0
assert np.allclose(x, expected_x, atol=5)
assert np.allclose(y, expected_y, atol=5)

Messing with WCS stuff to fix for `imagesim.py`. I ended up needing to use the same code that was used in the unit testing in `convert_pos` for the WCS from `test_image_sim.py` to put in the `run_simulation`function in `imagesim.py`. I think this helped since Galsim itself didn't support reading LONPOLE, but Galsim.Astropy did support reading it as a header.

In [ ]:
from astropy.wcs import WCS

w = WCS(myheader)

x_astropy_1, y_astropy_1 = w.world_to_pixel_values(
    10.2124400,
    -44.2785210,
)

print(x_astropy_1, y_astropy_1)

In [ ]:
x_astropy_2, y_astropy_2 = w.world_to_pixel_values(
    10.2950082,
    -44.3375139,
)

print(x_astropy_2, y_astropy_2)

In [ ]:
wcs_file_name = "/users/PCON0003/cond0007/PSF-TEST-FILES/Roman_WAS_simple_model_H158_13814_14.fits"
read_image = galsim.fits.read(file_name=wcs_file_name, hdu=1, read_header=True)
mywcs, neworigin = galsim.wcs.readFromFitsHeader(read_image.header)

print(mywcs)

In [ ]:
awcs = WCS(read_image.header)

print(awcs.pixel_scale_matrix)

This was some code to figure out another way to read yaml file with astropy for `imagesim.py`. This goes hand in hand with the code above. Again, didn't end up needing this because I didn't realize it was still using Galsim. I thought the solution was to just use Astropy, but it was much easier than that. 

In [ ]:
from astropy.io.misc import yaml

In [ ]:
# See if we can open and parse stand alone yaml file example_test.yaml for imagesim.py
with open(
    "/users/PAS2340/karadiludovico/sub_pixel_response/src/sub_pixel_response/example_test.yaml", "r"
) as f:
    yaml_data = yaml.load(f)

In [ ]:
# Getting LONPOLE value from example_test.yaml for imagesim.py
lonpole_value = yaml_data["LONPOLE"]
print("LONPOLE:", lonpole_value)

In [ ]:
# Also will try to get RA and Dec values from example_test.yaml for imagesim.py
# Not sure how to connect these with CRVAL1 and CRVAL2 values from given fits file used in imagesim.py
ra_value = yaml_data["raCen"]
dec_value = yaml_data["decCen"]
print("RA:", ra_value, "Dec:", dec_value)

In [ ]:
# For funnn, getting SCA value as well from example_test.yaml for imagesim.py
sca_value = yaml_data["SCA"]
print("SCA_NUM:", sca_value)

In [ ]:
myheader = fits.Header.fromstring(
    wcsstring,
    sep="\n",
)
mywcs = galsim.AstropyWCS(header=myheader)

This debugging is for figuring out the bounds for `j` since there are 32 tiles/tasks that are drawing the stars in the image. I'm trying to figure out how to connect `j` and `task_array` to each other. I will also have to figure out how to write `cat` based on this since the `RA` and `DEC` have to correspond to both. 

In [ ]:
from sub_pixel_response.imagesim import j_location

In [ ]:
for j in range(32):
    print(j, j_location(j))

In [ ]:
print(j_location(25))
print(j_location(26))
print(j_location(27))

Now, I've been trying to figure out different ways or ideas as to how to speed up the run time for `imagesim.py`. I want to see how many stars are being drawn in the image array, and how many stars are drawn in each of the 32 task arrays. This will show us the amount of stars that are drawn in a second. 

In [ ]:
from sub_pixel_response.simio import read_catalog, read_config
from sub_pixel_response.imagesim import convert_pos, assign_star

In [ ]:
config = read_config(
    "/users/PAS2340/karadiludovico/sub_pixel_response/src/sub_pixel_response/example_test.yaml"
)

In [ ]:
cat = read_catalog(config["starCat"])
nobj = len(cat["ra"])

In [ ]:
myheader = fits.Header.fromstring(
    wcsstring,
    sep="\n",
)
# mybounds = read_image.bounds
myheader["CRVAL1"] = float(config["raCen"])
myheader["CRVAL2"] = float(config["decCen"])
myheader["LONPOLE"] = float(config["LONPOLE"])
mywcs = galsim.AstropyWCS(header=myheader)

In [ ]:
from tqdm import tqdm

degrees = galsim.degrees
task_array = np.empty(nobj, dtype=np.int32)

for i in tqdm(range(nobj)):
    ra = cat["ra"][i] * degrees
    dec = cat["dec"][i] * degrees
    x, y = convert_pos(ra, dec, mywcs)
    task_array[i] = assign_star(x, y)

In [ ]:
counts = np.bincount(task_array)

print(counts)
print("Min:", counts.min())
print("Max:", counts.max())
print("Mean:", counts.mean())

In [ ]:
print(f"Min stars:  {counts.min():,}")
print(f"Max stars:  {counts.max():,}")
print(f"Mean stars: {counts.mean():,.0f}")

In [ ]:
print(np.sum(counts))

In [ ]:
print(counts)
print(np.unique(task_array))

The code above tells us that there are approximately 2.9 million stars being drawn out of the 32 task arrays. Out of the 32 task arrays, there are only 31 that are being used. This could be because of the image orientation that the task tile in the upper right corner isn't being used. This could also lead us to what is causing the most time usage in the code for the images to be drawn in 3.5 days. 

There are also a little over nine stars being drawn per second, which is still a lot, doing quick calculations for this. 

In [ ]:
xs = np.empty(nobj)
ys = np.empty(nobj)

for i in tqdm(range(nobj)):
    ra = cat["ra"][i] * galsim.degrees
    dec = cat["dec"][i] * galsim.degrees
    xs[i], ys[i] = convert_pos(ra, dec, mywcs)

Now, we are drawing out the task arrays to see what they look like. It's confirmed that the image array is being drawn in a diamond shape. We can aslo see which tasks are drawing the most stars in the image in the plot at the bottom. The orientation of the image explains why there is a random spread of stars being drawn in the image rather than them being spread out approximately the same between each of the 32 task arrays. 

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 8))
plt.scatter(xs, ys, s=0.05)
plt.xlabel("x")
plt.ylabel("y")
plt.axis("equal")
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(xs, ys, c=task_array, s=0.05, cmap="tab20")
plt.colorbar(label="Task")
plt.axis("equal")
plt.show()

Checking to see what takes up the most time in `draw_stars`. We're going to check for position, psf, flux, and drawImage.

In [ ]:
import cProfile
from sub_pixel_response.imagesim import transform_pos, compute_poly, sed_bb
from sub_pixel_response.utils.trapz import trapz
from astropy import constants as const
from astropy import units as u
import astropy.io as aio

In [ ]:
import time

t_position = 0
t_psf = 0
t_flux = 0
t_draw = 0

In [ ]:
t0 = time.perf_counter()

degrees = galsim.degrees
ra = cat["ra"][i] * degrees
dec = cat["dec"][i] * degrees

worldCenter = galsim.CelestialCoord(ra=ra, dec=dec)
imageCenter = mywcs.posToImage(worldCenter)

new_image_center = transform_pos(imageCenter.x, imageCenter.y)

imageCenter2 = galsim.PositionD(x=new_image_center[0], y=new_image_center[1])

t_position += time.perf_counter() - t0

print(t_position)

In [ ]:
t0 = time.perf_counter()

sca_num = int(config["SCA"])
psf_file = config["PSFFILE"]
worldCenter = galsim.CelestialCoord(ra=ra, dec=dec)
imageCenter = mywcs.posToImage(worldCenter)
new_image_center = transform_pos(imageCenter.x, imageCenter.y)
with fits.open(psf_file) as inpsf_file:
    psf_data = np.copy(inpsf_file[sca_num].data[:, :, :])
this_psf = compute_poly(psf_data, (new_image_center[0], new_image_center[1]))

t_psf += time.perf_counter() - t0

print(t_psf)

In [ ]:
t0 = time.perf_counter()

mag = cat["mag_H"][i]

t_exp = 120 * u.s
eff_area_table = aio.ascii.read(
    f"/users/PAS2340/karadiludovico/sub_pixel_response/src/sub_pixel_response/Roman_effarea_tables_20240327/Roman_effarea_v8_SCA{sca_num:02d}_20240301.ecsv"
)
filter_name = config["FILTER"]
mirror_diameter = 2.37 * u.m
geom_area = np.pi * mirror_diameter**2 / 4
transmission_curve = eff_area_table[filter_name] * u.m**2 / geom_area
wav = np.arange(0.400, 2.600, 0.001) * u.um
fluxUnnorm = sed_bb(wav, 5000 * u.K)
GLB_DATA = {
    "in_psf_oversam": 6,
    "f_nu_ref": 3.631e-23 * (u.W / u.m**2) / u.Hz,  # W/m^2/Hz
    "process_h": 4,
    "process_v": 8,
    "nside": 4088,
}
fLambdaRef = GLB_DATA["f_nu_ref"] * const.c / wav**2
norm = (
    10 ** (-0.4 * mag)
    * trapz(fLambdaRef * transmission_curve * wav, x=wav)
    / trapz(fluxUnnorm * transmission_curve * wav, x=wav)
)
flux = norm * fluxUnnorm
nPhotQ = trapz(
    flux * eff_area_table[filter_name] * u.m**2 * wav * t_exp / (const.h * const.c),
    x=wav,
)

t_flux += time.perf_counter() - t0

print(t_flux)

In [ ]:
t0 = time.perf_counter()

nPhotQ = nPhotQ.decompose()
nPhot = nPhotQ.value

st_model = galsim.DeltaFunction(flux=nPhot)

std_pad = 24
mybounds = j_location(j, x_padding=std_pad, y_padding=std_pad)
tempImage = galsim.Image(bounds=mybounds, dtype=np.float32)
big_fft_params = galsim.GSParams(maximum_fft_size=123000)
psf = galsim.Image(this_psf)
interp_psf = galsim.InterpolatedImage(psf, x_interpolant="lanczos32", scale=1)
source = galsim.Convolve([interp_psf, st_model], gsparams=big_fft_params)

source.drawImage(tempImage, method="no_pixel", center=imageCenter2, add_to_image=True)

t_draw += time.perf_counter() - t0

print(t_draw)

In [ ]:
print(f"Position : {t_position:.2f} s")
print(f"PSF      : {t_psf:.2f} s")
print(f"Flux     : {t_flux:.2f} s")
print(f"Draw     : {t_draw:.2f} s")

In [ ]:
indices = np.where(task_array == 14)[0][:1000]

for i in indices:
    if task_array[i] != j:
        continue

    if "is_in_circle" in cat and not cat["is_in_circle"][i]:
        continue

    t0 = time.perf_counter()

    degrees = galsim.degrees
    ra = cat["ra"][i] * degrees
    dec = cat["dec"][i] * degrees

    worldCenter = galsim.CelestialCoord(ra=ra, dec=dec)
    imageCenter = mywcs.posToImage(worldCenter)

    new_image_center = transform_pos(imageCenter.x, imageCenter.y)

    imageCenter2 = galsim.PositionD(x=new_image_center[0], y=new_image_center[1])

    t_position += time.perf_counter() - t0

    t0 = time.perf_counter()

    sca_num = int(config["SCA"])
    psf_file = config["PSFFILE"]
    worldCenter = galsim.CelestialCoord(ra=ra, dec=dec)
    imageCenter = mywcs.posToImage(worldCenter)
    new_image_center = transform_pos(imageCenter.x, imageCenter.y)
    with fits.open(psf_file) as inpsf_file:
        psf_data = np.copy(inpsf_file[sca_num].data[:, :, :])
    this_psf = compute_poly(psf_data, (new_image_center[0], new_image_center[1]))

    t_psf += time.perf_counter() - t0

    t0 = time.perf_counter()

    mag = cat["mag_H"][i]

    t_exp = 120 * u.s
    eff_area_table = aio.ascii.read(
        f"/users/PAS2340/karadiludovico/sub_pixel_response/src/sub_pixel_response/Roman_effarea_tables_20240327/Roman_effarea_v8_SCA{sca_num:02d}_20240301.ecsv"
    )
    filter_name = config["FILTER"]
    mirror_diameter = 2.37 * u.m
    geom_area = np.pi * mirror_diameter**2 / 4
    transmission_curve = eff_area_table[filter_name] * u.m**2 / geom_area
    wav = np.arange(0.400, 2.600, 0.001) * u.um
    fluxUnnorm = sed_bb(wav, 5000 * u.K)
    GLB_DATA = {
        "in_psf_oversam": 6,
        "f_nu_ref": 3.631e-23 * (u.W / u.m**2) / u.Hz,  # W/m^2/Hz
        "process_h": 4,
        "process_v": 8,
        "nside": 4088,
    }
    fLambdaRef = GLB_DATA["f_nu_ref"] * const.c / wav**2
    norm = (
        10 ** (-0.4 * mag)
        * trapz(fLambdaRef * transmission_curve * wav, x=wav)
        / trapz(fluxUnnorm * transmission_curve * wav, x=wav)
    )
    flux = norm * fluxUnnorm
    nPhotQ = trapz(
        flux * eff_area_table[filter_name] * u.m**2 * wav * t_exp / (const.h * const.c),
        x=wav,
    )

    t_flux += time.perf_counter() - t0

    t0 = time.perf_counter()

    nPhotQ = nPhotQ.decompose()
    nPhot = nPhotQ.value

    st_model = galsim.DeltaFunction(flux=nPhot)

    std_pad = 24
    mybounds = j_location(j, x_padding=std_pad, y_padding=std_pad)
    tempImage = galsim.Image(bounds=mybounds, dtype=np.float32)
    big_fft_params = galsim.GSParams(maximum_fft_size=123000)
    psf = galsim.Image(this_psf)
    interp_psf = galsim.InterpolatedImage(psf, x_interpolant="lanczos32", scale=1)
    source = galsim.Convolve([interp_psf, st_model], gsparams=big_fft_params)

    source.drawImage(tempImage, method="no_pixel", center=imageCenter2, add_to_image=True)

    t_draw += time.perf_counter() - t0

In [ ]:
print(f"Position : {t_position} s")
print(f"PSF      : {t_psf} s")
print(f"Flux     : {t_flux} s")
print(f"Draw     : {t_draw} s")

After all that, we can see that drawing the stars themselves using `source.drawImage` is taking the most time rather than the position, psf, and flux. 

Next I will check some other functions in the simulation to test if the runtimes are significant anywhere else in the code. 

In [ ]:
pipeline_times = {}
# Read catalog
t0 = time.perf_counter()

cat = read_catalog(config["starCat"])
pipeline_times["Read catalog"] = time.perf_counter() - t0

# Assign stars to tiles
t0 = time.perf_counter()

task_array = np.zeros(len(cat["ra"]), dtype=np.int32)
degrees = galsim.AngleUnit(np.pi / 180.0)

for i in range(len(cat["ra"])):
    ra = cat["ra"][i] * degrees
    dec = cat["dec"][i] * degrees
    x, y = convert_pos(ra, dec, mywcs)
    task_array[i] = assign_star(x, y)
pipeline_times["Assign stars"] = time.perf_counter() - t0

print()
for key, value in pipeline_times.items():
    print(f"{key}: {value:.3f} s")

For the draw_stars function, the PSF file is opened each time the function is called. This test will see if time can be saved from moving the file outside of the function. 

In [ ]:
# Inside the loop
t0 = time.perf_counter()

for i in range(20):
    with fits.open(psf_file) as f:
        psf = np.copy(f[sca_num].data)

print(time.perf_counter() - t0)

# Outside the loop
with fits.open(psf_file) as f:
    psf = np.copy(f[sca_num].data)

t0 = time.perf_counter()

for i in range(20):
    dummy = psf

print(time.perf_counter() - t0)

Some arrays of the draw_stars loop are constants such as the wavelength grid, 5000 K spectrum, and reference spectrum. This test will see how much time can be saved by moving them outside.

In [ ]:
# Current usage
N = 1000

t0 = time.perf_counter()

for i in range(N):
    wav = np.arange(0.400, 2.600, 0.001) * u.um

    fluxUnnorm = sed_bb(
        wav,
        5000 * u.K,
    )

    fLambdaRef = GLB_DATA["f_nu_ref"] * const.c / wav**2

elapsed_current = time.perf_counter() - t0

# Suggested usage
wav = np.arange(0.400, 2.600, 0.001) * u.um

fluxUnnorm = sed_bb(
    wav,
    5000 * u.K,
)

fLambdaRef = GLB_DATA["f_nu_ref"] * const.c / wav**2

t0 = time.perf_counter()

for i in range(N):
    pass

elapsed_cached = time.perf_counter() - t0

print(f"Current method : {elapsed_current:.3f} sec")
print(f"Cached method  : {elapsed_cached:.3f} sec")

print(f"Time saved: {elapsed_current-elapsed_cached:.3f} sec")


Now, we will divide the drawing of the image into different sections from the code to see how much of each takes up the most time. We will be looking at the run time for `st_model`, `source`, and `source.drawImage.`

In [ ]:
t_delta = 0
t_convolve = 0
t_draw = 0

In [ ]:
t0 = time.perf_counter()

st_model = galsim.DeltaFunction(flux=nPhot)

t_delta += time.perf_counter() - t0

In [ ]:
t0 = time.perf_counter()

source = galsim.Convolve([interp_psf, st_model], gsparams=big_fft_params)

t_convolve += time.perf_counter() - t0

In [ ]:
t0 = time.perf_counter()

source.drawImage(tempImage, method="no_pixel", center=imageCenter2, add_to_image=True)

t_draw = time.perf_counter() - t0

In [ ]:
print(f"Delta, st_model : {t_delta} s")
print(f"Convolve, source: {t_convolve} s")
print(f"source.drawImage: {t_draw} s")

Out of the three, the `source.drawImage` took the most time to run at 15 seconds. 

For this next run, we're going to see how big our `tempImage.bounds` is and the shape of the image array. 

In [ ]:
print(tempImage.bounds)
print(tempImage.array.shape)

Now below, we're going to use `psats` to see how long each part of `source.drawImage` takes and the information about what's called in that line of code in `imagesim.py`. 

In [ ]:
import pstats

prof = cProfile.Profile()

prof.enable()

source.drawImage(
    tempImage,
    method="no_pixel",
    center=imageCenter2,
    add_to_image=True,
)

prof.disable()

stats = pstats.Stats(prof)
stats.sort_stats("cumtime").print_stats(20)

Now, we're going to see how long indices are, which 1000 is to be expected. The line is doing what it's supposed to do and acts the same way as the for loop was originally written with `j` as before. 

In [ ]:
print(len(indices))

Now, we're going to print out the average time that it takes to draw a star in seconds. The output that we get below makes sense since there are over 9 stars being drawn per second out of 2.9 million stars. 

In [ ]:
print("Average draw time per star:", t_draw / len(indices), "seconds")

In [ ]:
print(this_psf.shape)

In [ ]:
print(this_psf.min())
print(this_psf.max())
print(np.sum(this_psf))

In [ ]:
plt.imshow(this_psf, origin="lower")
plt.colorbar()

In [ ]:
plt.imshow(np.log10(this_psf + 1e-20), origin="lower")
plt.colorbar()

In [ ]:
center = this_psf.shape[0] // 2

flux_128 = np.sum(this_psf[center - 64 : center + 64, center - 64 : center + 64])

flux_total = np.sum(this_psf)

print(flux_128 / flux_total)